# Exercise 13.3: KL Divergence and Asymmetry

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/Maths-for-IT-Notebooks/blob/main/Chapter-13/03-kl-divergence-asymmetry.ipynb)

**Book Reference:** Chapter 13 — Information Theory, Entropy and Loss Functions (`chapters/entropy-and-loss-functions.tex`, Section 2)  
**Estimated Time:** 15–20 min

### Learning Objectives
- Implement Kullback–Leibler divergence $D_{\mathrm{KL}}(P \| Q) = \sum p_i \log(p_i / q_i)$.
- Verify Gibbs' inequality ($D_{\mathrm{KL}}(P \| Q) \ge 0$) and the decomposition $H(P, Q) = H(P) + D_{\mathrm{KL}}(P \| Q)$.
- Demonstrate that KL divergence is asymmetric ($D_{\mathrm{KL}}(P \| Q) \neq D_{\mathrm{KL}}(Q \| P)$), unlike Jensen–Shannon divergence.


In [ ]:
import math
import numpy as np

### Mathematical Context

The **Kullback–Leibler (KL) divergence** measures the extra information cost of using model distribution $Q$ when the true distribution is $P$:
$$D_{\mathrm{KL}}(P \| Q) = H(P, Q) - H(P) = \sum_{i=1}^k p_i \log\frac{p_i}{q_i} \ge 0,$$
with equality if and only if $P = Q$.

Crucially, KL divergence is **not symmetric**: in general, $D_{\mathrm{KL}}(P \| Q) \neq D_{\mathrm{KL}}(Q \| P)$, so it is not a true distance metric.
A symmetric, bounded alternative is the **Jensen–Shannon divergence**:
$$D_{\mathrm{JS}}(P \| Q) = \frac{1}{2} D_{\mathrm{KL}}(P \| M) + \frac{1}{2} D_{\mathrm{KL}}(Q \| M), \quad M = \frac{1}{2}(P + Q).$$

In [ ]:
def kl_divergence(p: np.ndarray, q: np.ndarray, base: float | None = 2.0) -> float:
    """Compute KL divergence D_KL(P || Q) = sum_{p_i > 0} p_i * log(p_i / q_i).

    Args:
        p: 1D reference probability array.
        q: 1D approximating probability array.
        base: Logarithm base (default 2.0 for bits; None for nats).

    Returns:
        Non-negative KL divergence D_KL(P || Q).
    """
    # TODO: Validate p and q, check q[p > 0] > 0, and sum p[mask] * log(p[mask] / q[mask]).
    raise NotImplementedError("Complete this TODO")


def js_divergence(p: np.ndarray, q: np.ndarray, base: float = 2.0) -> float:
    """Compute symmetric Jensen-Shannon divergence JSD(P || Q).

    Formula:
        M = 0.5 * (P + Q)
        JSD(P || Q) = 0.5 * D_KL(P || M) + 0.5 * D_KL(Q || M)

    Args:
        p: 1D probability array.
        q: 1D probability array.
        base: Logarithm base (default 2.0).

    Returns:
        Symmetric JSD in [0, 1] when base=2.0.
    """
    # TODO: Compute M = 0.5 * (p + q) and return 0.5 * kl_divergence(p, M, base) + 0.5 * kl_divergence(q, M, base).
    raise NotImplementedError("Complete this TODO")

<details>
<summary>Hint 1: Safe KL computation</summary>

Just like Shannon entropy, mask with `mask = p_arr > 0.0`. Verify `np.all(q_arr[mask] > 0.0)` and compute `p_arr[mask] * np.log(p_arr[mask] / q_arr[mask])`.

</details>

<details>
<summary>Hint 2: Jensen-Shannon divergence</summary>

Notice that $M = 0.5 \cdot (P + Q)$ is strictly positive wherever either $P > 0$ or $Q > 0$, so `kl_divergence(P, M)` and `kl_divergence(Q, M)` are always well-defined!

</details>

In [ ]:
# Self-check assertions
p = np.array([0.9, 0.1])
q = np.array([0.5, 0.5])

# Identity: D_KL(P || P) == 0
assert abs(kl_divergence(p, p, base=2.0)) < 1e-12

# Asymmetry demonstration
kl_pq = kl_divergence(p, q, base=2.0)
kl_qp = kl_divergence(q, p, base=2.0)
assert kl_pq > 0.0 and kl_qp > 0.0
assert abs(kl_pq - kl_qp) > 0.1, "Expected D_KL(P||Q) != D_KL(Q||P)"

# Verify H(P, Q) == H(P) + D_KL(P || Q)
h_p = float(-np.sum(p * np.log2(p)))
h_pq = float(-np.sum(p * np.log2(q)))
assert abs(h_pq - (h_p + kl_pq)) < 1e-12

# Symmetry of JSD
assert abs(js_divergence(p, q) - js_divergence(q, p)) < 1e-12

print(f"D_KL(P || Q) = {kl_pq:.4f} bits")
print(f"D_KL(Q || P) = {kl_qp:.4f} bits (asymmetric!)")
print(f"JSD(P || Q)  = {js_divergence(p, q):.4f} bits (symmetric)")
print("All assertions passed!")

### Think Deeper

1. Why does minimizing $D_{\mathrm{KL}}(P \| Q)$ with respect to $Q$ force $Q$ to cover all modes of $P$ ("mode-covering"), whereas minimizing $D_{\mathrm{KL}}(Q \| P)$ tends to make $Q$ collapse onto a single mode of $P$ ("mode-seeking")?
2. Why is minimizing cross-entropy $H(P, Q)$ during neural network training equivalent to minimizing $D_{\mathrm{KL}}(P \| Q)$?